# Combined Datasets

ENTSO-E, JAO and Investing.com are joined into four datasets: trees and neural networks, each for Poland and for Poland plus the foreign addition. `PL_price` is the Polish day-ahead price at hour t and is the target. Columns follow the target, then ENTSO-E, JAO and Investing.com.

Investing.com has no Poland/foreign split, so its tree file is attached to both tree datasets and its neural-network file to both neural-network datasets. The foreign JAO files also hold the `pca_` inputs. They stay in the saved file. `load_final_dataset(..., with_pca_features=False)` drops every column whose name contains `pca`. That is the default.

The join is an inner join on the UTC index. The modeling window lies inside the overlap of the three sources. Files are written to `data/final/{model}_{scope}_{split}.parquet`. The download date range is not part of the file name.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

import pandas as pd

from src.config import (
    DOWNLOAD_END_DATE,
    DOWNLOAD_START_DATE,
    FULL_DATASET_END_DATE,
    FULL_DATASET_START_DATE,
)
from src.utils.feature_engineering.fe_calendar import (
    add_calendar_features,
    add_time_idx,
)
from src.utils.file_io import (
    final_dataset_path,
    inspect_merged_file,
    load_final_dataset,
    save_merged_data,
)
from src.utils.merging import (
    MODELS,
    SCOPES,
    cut_to_dates,
    merge_features,
    split_dataset,
    to_local_naive,
)

In [2]:
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"

## 1. Joining the Sources

Each dataset takes the feature file already saved for that model and scope. `PL_price` comes from the preprocessed Polish day-ahead price and is placed first.

NaN counts below are the lag warmup at the start of the download window. The modeling window starts later, once those lags exist.

In [3]:
merged = {}
for model in MODELS:
    for scope in SCOPES:
        df = merge_features(model, scope, date_range)
        merged[(model, scope)] = df
        print(f"{model}({scope})  {df.shape}  NaN {int(df.isna().sum().sum())}")

trees(pl)  (33239, 37)  NaN 1896
trees(pl_foreign)  (33239, 86)  NaN 2190
nn(pl)  (33239, 24)  NaN 603
nn(pl_foreign)  (33239, 69)  NaN 802


## 2. Local Index

The UTC index is hourly: one step, no duplicate timestamps, no missing hours. `to_local_naive` converts it to Europe/Warsaw and drops the timezone. Every UTC hour is kept. No hour is interpolated and no hour is dropped.

On the spring change the clock skips 02:00, so the index steps from 01:00 to 03:00. That hour does not exist. On the autumn change 02:00 happens twice and both hours stay. The later one is stored one second later, so the index is unique. Its `hour` value is still 2.

In [4]:
local = {key: to_local_naive(df) for key, df in merged.items()}

reference = local[("trees", "pl")]
steps = reference.index.to_series().diff().dropna()
irregular = steps[steps != pd.Timedelta(hours=1)]
same_index = all(df.index.equals(reference.index) for df in local.values())

print(f"rows {len(merged[('trees', 'pl')])} -> {len(reference)}")
print(
    f"tz {reference.index.tz}  unique {reference.index.is_unique}  same index {same_index}"
)
print(irregular)

rows 33239 -> 33239
tz None  unique True  same index True
2023-03-26 03:00:00   0 days 02:00:00
2023-10-29 02:00:01   0 days 00:00:01
2023-10-29 03:00:00   0 days 00:59:59
2024-03-31 03:00:00   0 days 02:00:00
2024-10-27 02:00:01   0 days 00:00:01
2024-10-27 03:00:00   0 days 00:59:59
2025-03-30 03:00:00   0 days 02:00:00
2025-10-26 02:00:01   0 days 00:00:01
2025-10-26 03:00:00   0 days 00:59:59
2026-03-29 03:00:00   0 days 02:00:00
dtype: timedelta64[us]


## 3. Modeling Window

`FULL_DATASET_START_DATE`–`FULL_DATASET_END_DATE` is the window the splits are cut from. It starts on 2022-12-26, after the lag warmup, and ends on 2026-09-13. Inside it the joined frame has no missing values.

In [5]:
window = {
    key: cut_to_dates(df, FULL_DATASET_START_DATE, FULL_DATASET_END_DATE)
    for key, df in local.items()
}

reference = window[("trees", "pl")]
same_index = all(df.index.equals(reference.index) for df in window.values())
print(f"{reference.index.min()} -> {reference.index.max()}")
print(
    f"{reference.shape}  NaN {int(reference.isna().sum().sum())}  same index {same_index}"
)

2022-12-26 00:00:00 -> 2026-09-13 23:00:00
(32591, 37)  NaN 0  same index True


## 4. Calendar Features

Built from the local index. The same five columns are attached to every dataset.

**Trees (LightGBM, XGBoost).** `hour`, `dow`, `month`, `is_holiday`, `is_bridge`. `dow` and `month` are integers. `is_holiday` and `is_bridge` are 0/1. No sin/cos encoding. `hour` stays in the frame so the rows can be split into 24 sets, one model per clock hour. It is not passed in as a feature: each model is trained for one hour. On a spring-change day there is no row with `hour` 2. On an autumn-change day there are two.

**Neural networks (TFT, LSTM).** The same five columns. `hour`, `dow` and `month` are integer category codes. No one-hot columns and no embeddings are built here. `is_holiday` and `is_bridge` are 0/1. `time_idx` counts consecutive steps from 0 at the first hour of the full window. It is not an input feature. Later hours of the same series already hold the calendar of the forecast horizon.

`dow` is Monday = 0 through Sunday = 6. `month` is 1–12. `hour` is 0–23. A holiday is a Polish public holiday, including 24 December from 2025. A bridge day is a working day with a day off on both sides: the Friday after a Thursday holiday, the Monday before a Tuesday holiday, or the one working day between two holidays. The flag is the same for every hour of that day.

In [6]:
ready = {}
for (model, scope), df in window.items():
    out = add_calendar_features(df)
    if model == "nn":
        out = add_time_idx(out)
    ready[(model, scope)] = out
    print(f"{model}({scope})  {out.shape}  {list(out.columns[-6:])}")

trees(pl)  (32591, 42)  ['de_baseload_chg7d', 'hour', 'dow', 'month', 'is_holiday', 'is_bridge']
trees(pl_foreign)  (32591, 91)  ['de_baseload_chg7d', 'hour', 'dow', 'month', 'is_holiday', 'is_bridge']
nn(pl)  (32591, 30)  ['hour', 'dow', 'month', 'is_holiday', 'is_bridge', 'time_idx']
nn(pl_foreign)  (32591, 75)  ['hour', 'dow', 'month', 'is_holiday', 'is_bridge', 'time_idx']


## 5. Train, Validation, Test

`SPLITS` reads the dates from `config.py`. `full` is the modeling window. `train`, `val` and `test` partition it: the next split starts one hour after the previous one ends. Trees and neural networks use the same hours. `time_idx` counts from the full window, so validation continues where training ends.

In [7]:
splits = {}
for (model, scope), df in ready.items():
    parts = split_dataset(df)
    splits[(model, scope)] = parts
    sizes = ", ".join(f"{name} {len(part)}" for name, part in parts.items())
    print(f"{model}({scope})  {sizes}  NaN {int(parts['full'].isna().sum().sum())}")

train = splits[("nn", "pl")]["train"]
val = splits[("nn", "pl")]["val"]
print(f"Index range: {train.index.max()} - {val.index.min()}")

trees(pl)  full 32591, train 17640, val 5879, test 9072  NaN 0
trees(pl_foreign)  full 32591, train 17640, val 5879, test 9072  NaN 0
nn(pl)  full 32591, train 17640, val 5879, test 9072  NaN 0
nn(pl_foreign)  full 32591, train 17640, val 5879, test 9072  NaN 0
Index range: 2024-12-29 23:00:00 - 2024-12-30 00:00:00


## 6. Saving

Sixteen files, four datasets times `full`, `train`, `val` and `test`, under `data/final/`. Trees and neural networks cover the same hours. None of the files contain missing values.

`load_final_dataset(model, scope, split, with_pca_features=False)` reads one file. `with_pca_features=True` keeps the `pca_` columns. Poland files have none either way.

In [8]:
for (model, scope), parts in splits.items():
    for name, part in parts.items():
        path = final_dataset_path(model, scope, name)
        save_merged_data(part, path)
        inspect_merged_file(path)

  [FILE] data/final/trees_pl_full.parquet | shape: (32591, 42) | idx: 2022-12-26 00:00:00-2026-09-13 23:00:00 | cols: 42 | NaN: 0
  [FILE] data/final/trees_pl_train.parquet | shape: (17640, 42) | idx: 2022-12-26 00:00:00-2024-12-29 23:00:00 | cols: 42 | NaN: 0
  [FILE] data/final/trees_pl_val.parquet | shape: (5879, 42) | idx: 2024-12-30 00:00:00-2025-08-31 23:00:00 | cols: 42 | NaN: 0
  [FILE] data/final/trees_pl_test.parquet | shape: (9072, 42) | idx: 2025-09-01 00:00:00-2026-09-13 23:00:00 | cols: 42 | NaN: 0
  [FILE] data/final/trees_pl_foreign_full.parquet | shape: (32591, 91) | idx: 2022-12-26 00:00:00-2026-09-13 23:00:00 | cols: 91 | NaN: 0
  [FILE] data/final/trees_pl_foreign_train.parquet | shape: (17640, 91) | idx: 2022-12-26 00:00:00-2024-12-29 23:00:00 | cols: 91 | NaN: 0
  [FILE] data/final/trees_pl_foreign_val.parquet | shape: (5879, 91) | idx: 2024-12-30 00:00:00-2025-08-31 23:00:00 | cols: 91 | NaN: 0
  [FILE] data/final/trees_pl_foreign_test.parquet | shape: (9072, 91)

In [9]:
pl = load_final_dataset("trees", "pl", "val")
pl_for = load_final_dataset("trees", "pl_foreign", "train")
pl_for_pca = load_final_dataset("trees", "pl_foreign", "train", with_pca_features=True)
print(pl.shape, "->", pl_for.shape, "->", pl_for_pca.shape)

(5879, 42) -> (17640, 67) -> (17640, 91)


In [10]:
pl = load_final_dataset("nn", "pl", "train")
pl_for = load_final_dataset("nn", "pl_foreign", "train")
pl_for_pca = load_final_dataset("nn", "pl_foreign", "train", with_pca_features=True)
print(pl.shape, "->", pl_for.shape, "->", pl_for_pca.shape)

(17640, 30) -> (17640, 51) -> (17640, 75)
